In [ ]:

!pip install transformers torch -q


In [ ]:

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed

set_seed(42)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Model loaded on:", device)


In [ ]:

def generate_text(prompt, max_new_tokens=60, **gen_kwargs):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        **gen_kwargs
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:

opening_sentence = "The old lighthouse keeper had never seen a storm like this before."

experiments = [
    {"name": "Baseline (greedy, do_sample=False)", "params": {"do_sample": False}},
    {"name": "Low Temperature (0.3)", "params": {"do_sample": True, "temperature": 0.3}},
    {"name": "High Temperature (1.5)", "params": {"do_sample": True, "temperature": 1.5}},
    {"name": "Top-k = 10", "params": {"do_sample": True, "temperature": 1.0, "top_k": 10}},
    {"name": "Top-p = 0.9 (nucleus sampling)", "params": {"do_sample": True, "temperature": 1.0, "top_p": 0.9}},
    {"name": "Repetition Penalty = 2.0", "params": {"do_sample": True, "temperature": 1.0, "repetition_penalty": 2.0}},
]

results = []
for exp in experiments:
    text = generate_text(opening_sentence, **exp["params"])
    results.append({"experiment": exp["name"], "params": exp["params"], "output": text})


In [ ]:

for r in results:
    print("=" * 90)
    print("Experiment:", r["experiment"])
    print("Parameters:", r["params"])
    print("-" * 90)
    print(r["output"])
    print()


In [ ]:

import pandas as pd

analysis_rows = []
for r in results:
    words = r["output"].split()
    unique_ratio = len(set(words)) / len(words) if words else 0
    analysis_rows.append({
        "Experiment": r["experiment"],
        "Word Count": len(words),
        "Unique Word Ratio": round(unique_ratio, 2)
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_df


In [ ]:

qualitative_analysis = '''
Qualitative Analysis of Parameter Effects:

- Baseline (greedy decoding): Produced the most repetitive and predictable text.
  Since the model always picks the single highest-probability token, sentences
  often loop into repeated phrases with little variation.

- Low Temperature (0.3): Very close to greedy decoding in behaviour. Output stayed
  coherent and "safe" but with limited creativity, since the probability
  distribution is sharpened toward the most likely tokens.

- High Temperature (1.5): Text became noticeably more random and, at times,
  incoherent. Rare/unlikely tokens were sampled more often, breaking grammatical
  flow and narrative logic.

- Top-k = 10: Restricting sampling to the 10 most probable tokens at each step
  produced a good balance -- more variety than greedy decoding, but still fairly
  coherent because implausible tokens were excluded.

- Top-p = 0.9 (nucleus sampling): Produced the most "creative but coherent" text
  among the experiments. By dynamically choosing the smallest set of tokens whose
  cumulative probability exceeds 0.9, it adapts to context better than a fixed top-k.

- Repetition Penalty = 2.0: Strongly discouraged the model from reusing words/phrases,
  reducing repetition, but pushed very high values toward slightly unnatural
  phrasing as the model avoided common connecting words.

Summary:
- Most repetitive/predictable: Baseline (greedy) and Low Temperature
- Most creative (coherent + varied): Top-p (nucleus sampling) and Top-k
- Most random/incoherent: High Temperature
'''
print(qualitative_analysis)


In [ ]:

custom_prompt = "Introducing the AeroFlow Smart Water Bottle, designed for those who never stop moving."

custom_experiments = [
    {"name": "Version 1: Balanced (Top-p sampling)", "params": {"do_sample": True, "top_p": 0.9, "temperature": 0.8}},
    {"name": "Version 2: Focused (Low temperature)", "params": {"do_sample": True, "temperature": 0.4}},
    {"name": "Version 3: Creative (High temperature + Top-k)", "params": {"do_sample": True, "temperature": 1.2, "top_k": 50}},
]

custom_results = []
for exp in custom_experiments:
    text = generate_text(custom_prompt, max_new_tokens=60, **exp["params"])
    custom_results.append({"experiment": exp["name"], "params": exp["params"], "output": text})

for r in custom_results:
    print("=" * 90)
    print("Experiment:", r["experiment"])
    print("Parameters:", r["params"])
    print("-" * 90)
    print(r["output"])
    print()


In [ ]:

report_summary = f'''
REPORT SUMMARY

Original Opening Sentence:
"{opening_sentence}"

Parameters Changed Across the 5 (+1 baseline) Experiments:
1. Baseline - greedy decoding (do_sample=False)
2. Temperature = 0.3 (low)
3. Temperature = 1.5 (high)
4. Top-k = 10
5. Top-p = 0.9 (nucleus sampling)
6. Repetition Penalty = 2.0

Findings:
- More repetitive/predictable text -> Baseline (greedy) and Low Temperature
- More creative text -> Top-p (nucleus sampling) and Top-k
- More random/incoherent text -> High Temperature

Custom Prompt:
"{custom_prompt}"

Three versions were generated using different parameter combinations
(balanced, focused, and creative), demonstrating how the same prompt can
produce very different outputs depending on the decoding strategy chosen.
'''
print(report_summary)
